In [ ]:
import numpy as np
from scipy.optimize import differential_evolution, minimize, brentq
from scipy.integrate import cumulative_trapezoid
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = './mock_outputs_v3'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📂 Résultats dans : {os.path.abspath(OUTPUT_DIR)}")

G_ASTRO = 4.302e-6
KPC_TO_CM = 3.086e21
MSUN_TO_G = 1.989e33
KM_TO_CM = 1e5
GYR_TO_S = 3.15576e16

# =============================================================================
# MODÈLE SIDM
# =============================================================================

class SIDMModel:
    def __init__(self, sigma_0: float, v_0: float, n: float = 4.0):
        self.sigma_0 = sigma_0
        self.v_0 = v_0
        self.n = n
        
    def sigma_over_m(self, v):
        return self.sigma_0 / (1.0 + (v / self.v_0)**self.n)
    
    def optical_depth(self, rho_msun_kpc3, v_kms, t_gyr):
        rho_cgs = rho_msun_kpc3 * MSUN_TO_G / (KPC_TO_CM**3)
        sigma_cgs = self.sigma_over_m(v_kms)
        v_cgs = v_kms * KM_TO_CM
        t_s = t_gyr * GYR_TO_S
        return rho_cgs * sigma_cgs * v_cgs * t_s
    
    def thermalization_radius(self, rho_s, r_s, t_age=10.0):
        def condition(r):
            if r <= 0:
                return -1e10
            x = r / r_s
            rho = rho_s / (x * (1 + x)**2 + 1e-10)
            M_enc = 4 * np.pi * rho_s * r_s**3 * (np.log(1 + x) - x/(1+x))
            v = np.sqrt(G_ASTRO * M_enc / (r + 1e-10))
            v = np.clip(v, 10, 500)
            tau = self.optical_depth(rho, v, t_age)
            return tau - 1.0
        
        try:
            r_min, r_max = 0.01 * r_s, 5.0 * r_s
            f_min, f_max = condition(r_min), condition(r_max)
            if f_min * f_max > 0:
                return 0.5 * r_s if f_min < 0 else 2.0 * r_s
            return np.clip(brentq(condition, r_min, r_max), 0.05, 20.0)
        except:
            return 0.5 * r_s
    
    def density_profile(self, r, rho_s, r_s, t_age=10.0):
        r_1 = self.thermalization_radius(rho_s, r_s, t_age)
        r_core = r_1
        x_1 = r_1 / r_s
        M_nfw_r1 = 4 * np.pi * rho_s * r_s**3 * (np.log(1 + x_1) - x_1 / (1 + x_1))
        rho_0 = M_nfw_r1 / (4/3 * np.pi * r_1**3) * 1.5
        rho_0 = min(rho_0, 10 * rho_s)
        rho_isothermal = rho_0 / (1 + (r / r_core)**2)
        x = r / r_s
        rho_nfw = rho_s / (x * (1 + x)**2 + 1e-10)
        r_match = 2.0 * r_core
        transition = 0.5 * (1 + np.tanh((r - r_match) / (0.5 * r_core)))
        return (1 - transition) * rho_isothermal + transition * rho_nfw, r_core
    
    def rotation_curve(self, r, rho_s, r_s, t_age=10.0):
        r_fine = np.logspace(np.log10(max(r.min(), 0.01)), 
                             np.log10(r.max() * 1.5), 500)
        rho, r_core = self.density_profile(r_fine, rho_s, r_s, t_age)
        integrand = 4 * np.pi * r_fine**2 * rho
        M_enc = cumulative_trapezoid(integrand, r_fine, initial=0)
        if len(M_enc) > 1:
            M_enc[0] = M_enc[1] * (r_fine[0] / r_fine[1])**3
        V_fine = np.sqrt(G_ASTRO * np.maximum(M_enc, 0) / np.maximum(r_fine, 0.01))
        V_dm = np.interp(r, r_fine, V_fine)
        return V_dm, r_core


# =============================================================================
# COMPOSANTES BARYONIQUES
# =============================================================================

def disk_velocity_freeman(r, M_disk, R_disk):
    from scipy.special import i0, i1, k0, k1
    y = np.clip(r / (2.0 * R_disk), 1e-10, 100)
    Sigma_0 = M_disk / (2.0 * np.pi * R_disk**2)
    bessel_term = i0(y) * k0(y) - i1(y) * k1(y)
    V2 = 4.0 * np.pi * G_ASTRO * Sigma_0 * R_disk * y**2 * bessel_term
    return np.sqrt(np.maximum(V2, 0))

def gas_velocity(r, M_gas, R_gas):
    return disk_velocity_freeman(r, M_gas, R_gas)


# =============================================================================
# GÉNÉRATION MOCK (identique)
# =============================================================================

def compute_vmax_for_template(template, sidm_model):
    L_disk = template['L_disk']
    R_disk = template['R_disk']
    M_gas = template['M_gas']
    R_gas = template['R_gas']
    rho_s = template['rho_s']
    r_s = template['r_s']
    Y_disk = template['Y_disk']
    
    r_max = max(6 * R_disk, 4 * r_s, 30)
    r = np.linspace(0.3, r_max, 200)
    
    M_disk = L_disk * Y_disk
    V_disk = disk_velocity_freeman(r, M_disk, R_disk)
    V_gas = gas_velocity(r, M_gas, R_gas)
    V_dm, _ = sidm_model.rotation_curve(r, rho_s, r_s)
    
    V_tot = np.sqrt(V_disk**2 + V_gas**2 + V_dm**2)
    return np.max(V_tot)


def generate_mock_galaxy(name, L_disk, R_disk, M_gas, R_gas,
                         rho_s_true, r_s_true, Y_disk_true,
                         sidm_model, n_points=25, error_frac=0.08,
                         seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    r_min = 0.3
    r_max = min(6 * R_disk, 8 * r_s_true)
    r_max = max(r_max, 5.0)
    r = np.linspace(r_min, r_max, n_points)
    
    M_disk = L_disk * Y_disk_true
    V_disk = disk_velocity_freeman(r, M_disk, R_disk)
    V_gas = gas_velocity(r, M_gas, R_gas)
    V_bar = np.sqrt(V_disk**2 + V_gas**2)
    
    V_dm, r_core = sidm_model.rotation_curve(r, rho_s_true, r_s_true)
    V_true = np.sqrt(V_bar**2 + V_dm**2)
    
    V_err = np.maximum(V_true * error_frac, 2.0)
    V_obs = V_true + np.random.normal(0, V_err)
    V_obs = np.maximum(V_obs, 5.0)
    
    return {
        'name': name,
        'r': r,
        'V_obs': V_obs,
        'V_err': V_err,
        'V_true': V_true,
        'V_disk': V_disk,
        'V_gas': V_gas,
        'V_dm': V_dm,
        'r_core_true': r_core,
        'true_params': {
            'rho_s': rho_s_true,
            'r_s': r_s_true,
            'Y_disk': Y_disk_true,
            'Y_bulge': 0.7
        },
        'galaxy_props': {
            'L_disk': L_disk,
            'R_disk': R_disk,
            'M_gas': M_gas,
            'R_gas': R_gas
        }
    }


def create_realistic_mock_sample(sigma_0_true, v_0_true, n_galaxies=30):
    sidm = SIDMModel(sigma_0_true, v_0_true)
    
    galaxy_templates = [
        # ULTRA-NAINES
        {'L_disk': 1e6,  'R_disk': 0.4, 'M_gas': 1e7,  'R_gas': 0.8, 
         'rho_s': 2e8,   'r_s': 0.8,   'Y_disk': 0.5, 'n_points': 8, 'category': 'ultra_dwarf'},
        {'L_disk': 2e6,  'R_disk': 0.5, 'M_gas': 2e7,  'R_gas': 1.0, 
         'rho_s': 1.5e8, 'r_s': 1.0,   'Y_disk': 0.5, 'n_points': 8, 'category': 'ultra_dwarf'},
        {'L_disk': 5e6,  'R_disk': 0.6, 'M_gas': 4e7,  'R_gas': 1.2, 
         'rho_s': 1e8,   'r_s': 1.2,   'Y_disk': 0.45, 'n_points': 10, 'category': 'ultra_dwarf'},
        # NAINES
        {'L_disk': 1e7,  'R_disk': 0.8, 'M_gas': 8e7,  'R_gas': 1.6, 
         'rho_s': 6e7,   'r_s': 1.8,   'Y_disk': 0.5, 'n_points': 12, 'category': 'dwarf'},
        {'L_disk': 2e7,  'R_disk': 1.0, 'M_gas': 1.5e8,'R_gas': 2.0, 
         'rho_s': 4e7,   'r_s': 2.5,   'Y_disk': 0.45, 'n_points': 12, 'category': 'dwarf'},
        {'L_disk': 5e7,  'R_disk': 1.2, 'M_gas': 3e8,  'R_gas': 2.5, 
         'rho_s': 2.5e7, 'r_s': 3.0,   'Y_disk': 0.5, 'n_points': 14, 'category': 'dwarf'},
        # LSB
        {'L_disk': 1e8,  'R_disk': 1.8, 'M_gas': 6e8,  'R_gas': 3.5, 
         'rho_s': 1.5e7, 'r_s': 4.5,   'Y_disk': 0.45, 'n_points': 16, 'category': 'lsb'},
        {'L_disk': 3e8,  'R_disk': 2.5, 'M_gas': 1.2e9,'R_gas': 5.0, 
         'rho_s': 1e7,   'r_s': 6.0,   'Y_disk': 0.4, 'n_points': 18, 'category': 'lsb'},
        {'L_disk': 6e8,  'R_disk': 3.0, 'M_gas': 2e9,  'R_gas': 6.0, 
         'rho_s': 7e6,   'r_s': 8.0,   'Y_disk': 0.45, 'n_points': 20, 'category': 'lsb'},
        # SPIRALES
        {'L_disk': 2e9,  'R_disk': 3.5, 'M_gas': 3e9,  'R_gas': 7.0, 
         'rho_s': 5e6,   'r_s': 10.0,  'Y_disk': 0.5, 'n_points': 22, 'category': 'spiral'},
        {'L_disk': 5e9,  'R_disk': 4.0, 'M_gas': 4e9,  'R_gas': 8.0, 
         'rho_s': 4e6,   'r_s': 12.0,  'Y_disk': 0.55, 'n_points': 25, 'category': 'spiral'},
        {'L_disk': 1e10, 'R_disk': 4.5, 'M_gas': 5e9,  'R_gas': 9.0, 
         'rho_s': 3.5e6, 'r_s': 14.0,  'Y_disk': 0.5, 'n_points': 28, 'category': 'spiral'},
        # MASSIVES
        {'L_disk': 2e10, 'R_disk': 5.0, 'M_gas': 8e9,  'R_gas': 10.0, 
         'rho_s': 4e6,   'r_s': 18.0,  'Y_disk': 0.55, 'n_points': 30, 'category': 'massive'},
        {'L_disk': 4e10, 'R_disk': 6.0, 'M_gas': 1.2e10,'R_gas': 12.0, 
         'rho_s': 5e6,   'r_s': 22.0,  'Y_disk': 0.5, 'n_points': 32, 'category': 'massive'},
        # TRÈS MASSIVES
        {'L_disk': 8e10, 'R_disk': 7.0, 'M_gas': 2e10, 'R_gas': 14.0, 
         'rho_s': 6e6,   'r_s': 28.0,  'Y_disk': 0.55, 'n_points': 35, 'category': 'very_massive'},
        {'L_disk': 1.5e11,'R_disk': 9.0, 'M_gas': 3e10, 'R_gas': 18.0, 
         'rho_s': 7e6,   'r_s': 35.0,  'Y_disk': 0.6, 'n_points': 40, 'category': 'very_massive'},
        {'L_disk': 2.5e11,'R_disk': 11.0,'M_gas': 4e10, 'R_gas': 22.0, 
         'rho_s': 8e6,   'r_s': 42.0,  'Y_disk': 0.6, 'n_points': 45, 'category': 'very_massive'},
    ]
    
    galaxies = []
    for i in range(n_galaxies):
        template = galaxy_templates[i % len(galaxy_templates)].copy()
        category = template.pop('category', 'unknown')
        
        np.random.seed(42 + i)
        for key in ['L_disk', 'M_gas', 'rho_s', 'r_s']:
            template[key] *= np.random.uniform(0.8, 1.2)
        template['Y_disk'] = np.clip(
            template['Y_disk'] + np.random.uniform(-0.1, 0.1), 0.2, 0.9
        )
        
        galaxy = generate_mock_galaxy(
            name=f"Mock_{i+1:03d}_{category}",
            L_disk=template['L_disk'],
            R_disk=template['R_disk'],
            M_gas=template['M_gas'],
            R_gas=template['R_gas'],
            rho_s_true=template['rho_s'],
            r_s_true=template['r_s'],
            Y_disk_true=template['Y_disk'],
            sidm_model=sidm,
            n_points=template['n_points'],
            error_frac=np.random.uniform(0.05, 0.10),
            seed=42 + i
        )
        galaxies.append(galaxy)
    
    return galaxies


# =============================================================================
# FITTER CORRIGÉ - PAS DE STAGNATION
# =============================================================================

def chi2_single_galaxy(local_params, galaxy, sigma_0, v_0):
    """Calcule χ² pour une galaxie avec paramètres locaux donnés."""
    log_rho_s, log_r_s, Y_disk, Y_bulge = local_params
    
    if not (4.0 < log_rho_s < 10.0 and -1.0 < log_r_s < 2.5 and 0.1 < Y_disk < 1.5):
        return 1e10
    
    rho_s = 10**log_rho_s
    r_s = 10**log_r_s
    
    r = galaxy['r']
    V_obs = galaxy['V_obs']
    V_err = galaxy['V_err']
    props = galaxy['galaxy_props']
    
    M_disk = props['L_disk'] * Y_disk
    V_disk = disk_velocity_freeman(r, M_disk, props['R_disk'])
    V_gas = gas_velocity(r, props['M_gas'], props['R_gas'])
    
    sidm = SIDMModel(sigma_0, v_0)
    V_dm, _ = sidm.rotation_curve(r, rho_s, r_s)
    
    V_model = np.sqrt(V_disk**2 + V_gas**2 + V_dm**2)
    chi2 = np.sum(((V_obs - V_model) / V_err)**2)
    
    return chi2 if np.isfinite(chi2) else 1e10


def fit_single_galaxy(galaxy, sigma_0, v_0, init_guess=None):
    """
    Fit une galaxie avec paramètres globaux fixés.
    Utilise multi-start pour éviter minima locaux.
    """
    local_bounds = [
        (4.5, 9.5),    # log_rho_s
        (-0.5, 2.2),   # log_r_s  
        (0.15, 1.2),   # Y_disk
        (0.3, 1.2)     # Y_bulge
    ]
    
    best_chi2 = 1e20
    best_params = None
    
    # Multi-start : plusieurs points de départ
    if init_guess is not None:
        starts = [init_guess]
    else:
        starts = []
    
    # Ajouter des points de départ basés sur les vraies valeurs (si mock)
    # et des points génériques
    starts.extend([
        [7.0, 0.5, 0.5, 0.7],   # Générique
        [6.5, 1.0, 0.4, 0.7],   # Dwarf-like
        [8.0, 0.3, 0.6, 0.7],   # Dense
        [6.0, 1.5, 0.5, 0.7],   # Étendu
    ])
    
    for start in starts:
        try:
            result = minimize(
                lambda p: chi2_single_galaxy(p, galaxy, sigma_0, v_0),
                start,
                method='L-BFGS-B',
                bounds=local_bounds,
                options={'ftol': 1e-6, 'maxiter': 200}
            )
            
            if result.fun < best_chi2:
                best_chi2 = result.fun
                best_params = result.x.copy()
        except:
            continue
    
    if best_params is None:
        best_params = np.array([7.0, 0.5, 0.5, 0.7])
        best_chi2 = chi2_single_galaxy(best_params, galaxy, sigma_0, v_0)
    
    return best_params, best_chi2


def compute_total_chi2(sigma_0, v_0, galaxies, return_params=False):
    """
    Calcule le χ² total en optimisant COMPLÈTEMENT les paramètres locaux.
    C'est la clé : on repart de zéro à chaque évaluation.
    """
    total_chi2 = 0.0
    all_params = []
    
    for gal in galaxies:
        params, chi2 = fit_single_galaxy(gal, sigma_0, v_0)
        total_chi2 += chi2
        all_params.append(params)
    
    if return_params:
        return total_chi2, all_params
    return total_chi2


def fit_global_sidm_grid_then_refine(galaxies, verbose=True):
    """
    STRATÉGIE ANTI-STAGNATION:
    1. Grid search grossier pour trouver la bonne région
    2. Raffinement local autour du meilleur point
    """
    if verbose:
        print("\n" + "="*60)
        print("SIDM FIT - Grid Search + Raffinement")
        print("="*60)
    
    # ===================
    # PHASE 1: GRID SEARCH
    # ===================
    if verbose:
        print("\n[Phase 1] Grid search...")
    
    sigma_grid = [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 15.0, 25.0]
    v0_grid = [15, 25, 35, 50, 70, 100, 150]
    
    best_chi2 = 1e20
    best_sigma = 3.0
    best_v0 = 35.0
    
    results_grid = []
    
    for sigma_0 in sigma_grid:
        for v_0 in v0_grid:
            chi2 = compute_total_chi2(sigma_0, v_0, galaxies)
            results_grid.append((sigma_0, v_0, chi2))
            
            if chi2 < best_chi2:
                best_chi2 = chi2
                best_sigma = sigma_0
                best_v0 = v_0
            
            if verbose:
                print(f"    σ₀={sigma_0:5.1f}, v₀={v_0:5.0f} → χ²={chi2:8.1f}" + 
                      (" *" if chi2 == best_chi2 else ""))
    
    if verbose:
        print(f"\n  Meilleur point grille: σ₀={best_sigma:.1f}, v₀={best_v0:.0f}, χ²={best_chi2:.1f}")
    
    # ===================
    # PHASE 2: RAFFINEMENT LOCAL
    # ===================
    if verbose:
        print("\n[Phase 2] Raffinement (Nelder-Mead)...")
    
    def objective(params):
        sigma_0, v_0 = params
        if sigma_0 < 0.1 or sigma_0 > 50 or v_0 < 10 or v_0 > 200:
            return 1e15
        return compute_total_chi2(sigma_0, v_0, galaxies)
    
    # Plusieurs départs autour du meilleur point de la grille
    best_refined = None
    best_refined_chi2 = best_chi2
    
    start_points = [
        [best_sigma, best_v0],
        [best_sigma * 0.7, best_v0 * 0.8],
        [best_sigma * 1.3, best_v0 * 1.2],
        [best_sigma * 0.8, best_v0 * 1.3],
        [best_sigma * 1.2, best_v0 * 0.7],
    ]
    
    for i, start in enumerate(start_points):
        if verbose:
            print(f"  Start {i+1}: σ₀={start[0]:.2f}, v₀={start[1]:.1f}")
        
        try:
            result = minimize(
                objective,
                start,
                method='Nelder-Mead',
                options={'maxiter': 100, 'xatol': 0.1, 'fatol': 5.0}
            )
            
            if verbose:
                print(f"    → σ₀={result.x[0]:.2f}, v₀={result.x[1]:.1f}, χ²={result.fun:.1f}")
            
            if result.fun < best_refined_chi2:
                best_refined_chi2 = result.fun
                best_refined = result.x.copy()
        except Exception as e:
            if verbose:
                print(f"    → Échec: {e}")
    
    if best_refined is None:
        best_refined = np.array([best_sigma, best_v0])
    
    sigma_0_fit, v_0_fit = best_refined
    
    # ===================
    # PHASE 3: FIT FINAL
    # ===================
    if verbose:
        print("\n[Phase 3] Fits finaux des galaxies...")
    
    final_chi2, final_params = compute_total_chi2(sigma_0_fit, v_0_fit, galaxies, return_params=True)
    
    n_dof = sum(len(g['r']) - 4 for g in galaxies) - 2
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"RÉSULTATS:")
        print(f"  σ₀ = {sigma_0_fit:.3f} cm²/g")
        print(f"  v₀ = {v_0_fit:.2f} km/s")
        print(f"  χ² = {final_chi2:.1f} / {n_dof} = {final_chi2/n_dof:.3f}")
        print(f"{'='*60}")
    
    return sigma_0_fit, v_0_fit, final_params, final_chi2, n_dof


# =============================================================================
# VISUALISATION
# =============================================================================

def plot_results(galaxies, sigma_0_fit, v_0_fit, final_params,
                 sigma_0_true, v_0_true, filename='mock_results_v3.png'):
    
    Vmax_list = [(i, np.max(g['V_true'])) for i, g in enumerate(galaxies)]
    Vmax_list.sort(key=lambda x: x[1])
    
    n_show = min(9, len(galaxies))
    step = max(1, len(Vmax_list) // n_show)
    indices_to_show = [Vmax_list[min(i * step, len(Vmax_list)-1)][0] for i in range(n_show)]
    
    fig = plt.figure(figsize=(15, 16))
    sidm_fit = SIDMModel(sigma_0_fit, v_0_fit)
    
    for plot_idx, gal_idx in enumerate(indices_to_show):
        ax = fig.add_subplot(4, 3, plot_idx + 1)
        gal = galaxies[gal_idx]
        params = final_params[gal_idx]
        r = gal['r']
        
        ax.errorbar(r, gal['V_obs'], yerr=gal['V_err'], fmt='ko', ms=3,
                    capsize=1, label='Data', zorder=5)
        ax.plot(r, gal['V_true'], 'b--', lw=2, alpha=0.7, label='True')
        
        rho_s, r_s = 10**params[0], 10**params[1]
        Y_disk = params[2]
        M_disk = gal['galaxy_props']['L_disk'] * Y_disk
        V_disk = disk_velocity_freeman(r, M_disk, gal['galaxy_props']['R_disk'])
        V_gas = gas_velocity(r, gal['galaxy_props']['M_gas'], gal['galaxy_props']['R_gas'])
        V_dm, _ = sidm_fit.rotation_curve(r, rho_s, r_s)
        V_fit = np.sqrt(V_disk**2 + V_gas**2 + V_dm**2)
        
        ax.plot(r, V_fit, 'r-', lw=2, label='Fit')
        
        Vmax = np.max(gal['V_true'])
        chi2 = np.sum(((gal['V_obs'] - V_fit) / gal['V_err'])**2)
        chi2_red = chi2 / max(len(r) - 4, 1)
        
        ax.set_xlabel('R (kpc)')
        ax.set_ylabel('V (km/s)')
        ax.set_title(f"Vmax={Vmax:.0f} km/s, χ²/dof={chi2_red:.2f}")
        ax.legend(fontsize=7, loc='lower right')
        ax.set_xlim(0, None)
        ax.set_ylim(0, None)
    
    # Résumé
    ax = fig.add_subplot(4, 3, 10)
    ax.axis('off')
    
    delta_sigma = 100 * (sigma_0_fit - sigma_0_true) / sigma_0_true
    delta_v = 100 * (v_0_fit - v_0_true) / v_0_true
    success = abs(delta_sigma) < 20 and abs(delta_v) < 20
    
    summary = f"""
    MOCK TEST RESULTS
    =================
    
    TRUE:  σ₀ = {sigma_0_true:.2f} cm²/g
           v₀ = {v_0_true:.1f} km/s
    
    FIT:   σ₀ = {sigma_0_fit:.2f} cm²/g  ({delta_sigma:+.1f}%)
           v₀ = {v_0_fit:.1f} km/s  ({delta_v:+.1f}%)
    
    {'✓ SUCCESS' if success else '✗ FAILED'}
    """
    
    color = 'lightgreen' if success else 'lightsalmon'
    ax.text(0.05, 0.95, summary, transform=ax.transAxes, fontsize=12,
           verticalalignment='top', fontfamily='monospace',
           bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))
    
    # σ(v) curve
    ax = fig.add_subplot(4, 3, 11)
    v_range = np.logspace(1, 2.5, 100)
    sidm_true = SIDMModel(sigma_0_true, v_0_true)
    ax.loglog(v_range, sidm_true.sigma_over_m(v_range), 'b-', lw=2, label='True')
    ax.loglog(v_range, sidm_fit.sigma_over_m(v_range), 'r--', lw=2, label='Fit')
    Vmax_all = [np.max(g['V_true']) for g in galaxies]
    ax.axvspan(min(Vmax_all), max(Vmax_all), alpha=0.2, color='green', label='Data range')
    ax.set_xlabel('v (km/s)')
    ax.set_ylabel('σ/m (cm²/g)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Histogram
    ax = fig.add_subplot(4, 3, 12)
    ax.hist(Vmax_all, bins=12, edgecolor='black', alpha=0.7)
    ax.axvline(v_0_true, color='r', ls='--', label=f'v₀={v_0_true}')
    ax.set_xlabel('Vmax (km/s)')
    ax.set_ylabel('Count')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150)
    plt.show()
    print(f"\n✓ Figure sauvegardée: {os.path.join(OUTPUT_DIR, filename)}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("="*60)
    print("SIDM MOCK TEST - V3 (Anti-Stagnation)")
    print("="*60)
    
    SIGMA_0_TRUE = 3.27
    V_0_TRUE = 34.9
    N_GALAXIES = 36
    
    print(f"\nParamètres vrais: σ₀={SIGMA_0_TRUE}, v₀={V_0_TRUE}")
    print(f"Génération de {N_GALAXIES} galaxies mock...")
    
    galaxies = create_realistic_mock_sample(SIGMA_0_TRUE, V_0_TRUE, N_GALAXIES)
    
    Vmax_values = [np.max(g['V_true']) for g in galaxies]
    print(f"Vmax: {min(Vmax_values):.0f} - {max(Vmax_values):.0f} km/s")
    
    # FIT
    sigma_0_fit, v_0_fit, final_params, chi2_total, n_dof = fit_global_sidm_grid_then_refine(
        galaxies, verbose=True
    )
    
    # Résumé
    delta_s = 100 * (sigma_0_fit - SIGMA_0_TRUE) / SIGMA_0_TRUE
    delta_v = 100 * (v_0_fit - V_0_TRUE) / V_0_TRUE
    success = abs(delta_s) < 20 and abs(delta_v) < 20
    
    print(f"\n{'='*60}")
    print("RÉSUMÉ FINAL")
    print(f"{'='*60}")
    print(f"  σ₀: {SIGMA_0_TRUE:.2f} → {sigma_0_fit:.2f} [{delta_s:+.1f}%]")
    print(f"  v₀: {V_0_TRUE:.1f} → {v_0_fit:.1f} [{delta_v:+.1f}%]")
    print(f"\n{'✓ MOCK TEST PASSED' if success else '✗ MOCK TEST FAILED'}")
    
    # Figure
    plot_results(galaxies, sigma_0_fit, v_0_fit, final_params,
                 SIGMA_0_TRUE, V_0_TRUE)
    
    return sigma_0_fit, v_0_fit


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
"""
SIDM Mock Test - Version Corrigée
==================================
Ce mock test est maintenant cohérent avec l'analyse SPARC :
- Les M/L ratios (Y_disk, Y_bulge) sont des paramètres LIBRES du fit
- Même structure de fit que le code principal

Auteur: Florian Celli (corrigé par Claude)
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, brentq
from scipy.integrate import cumulative_trapezoid
from scipy.special import i0, i1, k0, k1
from dataclasses import dataclass
from typing import Dict, Tuple, List
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
import warnings
import os
import time
import json

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
OUTPUT_DIR = './mock_test_corrected_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_WORKERS = min(mp.cpu_count() - 1, 80)

class PhysicalConstants:
    G = 4.302e-6
    kpc_to_cm = 3.086e21
    Msun_to_g = 1.989e33
    km_to_cm = 1e5
    Gyr_to_s = 3.15576e16
    t_Hubble = 13.8

PC = PhysicalConstants()


@dataclass
class GalaxyData:
    name: str
    radius: np.ndarray
    Vobs: np.ndarray
    errV: np.ndarray
    Vgas: np.ndarray
    Vdisk: np.ndarray
    Vbul: np.ndarray
    distance: float = 10.0
    inclination: float = 60.0
    luminosity: float = 1e9
    morphology: str = ""
    quality: int = 1
    Y_disk_true: float = 0.5
    Y_bulge_true: float = 0.7
    
    @property
    def n_points(self) -> int:
        return len(self.radius)
    
    @property
    def Vflat(self) -> float:
        return np.median(self.Vobs[-5:]) if len(self.Vobs) >= 5 else self.Vobs[-1]
    
    @property
    def Vmax(self) -> float:
        return np.max(self.Vobs)


class ScalingRelations:
    @staticmethod
    def Mhalo_from_Vflat(Vflat: float) -> float:
        log_Mh = 10.0 + 3.5 * np.log10(Vflat / 100)
        return 10**np.clip(log_Mh, 8.0, 15.0)
    
    @staticmethod
    def concentration(Mhalo: float) -> float:
        log_Mh = np.log10(Mhalo)
        a = 0.520 + (0.905 - 0.520) * np.exp(-0.617 * 0**1.21)
        b = -0.101
        log_c = a + b * (log_Mh - 12)
        return 10**np.clip(log_c, 0.3, 1.8)
    
    @staticmethod
    def r_vir(Mhalo: float, Delta: float = 200) -> float:
        rho_crit = 127.7
        return (3 * Mhalo / (4 * np.pi * Delta * rho_crit))**(1/3)
    
    @staticmethod
    def r_s(Mhalo: float) -> float:
        c = ScalingRelations.concentration(Mhalo)
        r_200 = ScalingRelations.r_vir(Mhalo, Delta=200)
        return r_200 / c
    
    @staticmethod
    def rho_s(Mhalo: float) -> float:
        c = ScalingRelations.concentration(Mhalo)
        r_s = ScalingRelations.r_s(Mhalo)
        f_c = np.log(1 + c) - c / (1 + c)
        return Mhalo / (4 * np.pi * r_s**3 * f_c)
    
    @staticmethod
    def halo_age(Mhalo: float) -> float:
        z_f = 2.5 * (Mhalo / 1e12)**(-0.15)
        z_f = np.clip(z_f, 0.5, 6.0)
        t_form = PC.t_Hubble * (1 + z_f)**(-1.5)
        return PC.t_Hubble - t_form


class SIDMModel:
    def __init__(self, sigma_0: float = 50.0, v_0: float = 50.0, n: float = 4.0):
        self.sigma_0 = sigma_0
        self.v_0 = v_0
        self.n = n
        
    def sigma_over_m(self, v):
        return self.sigma_0 / (1.0 + (v / self.v_0)**self.n)
    
    def optical_depth(self, rho_msun_per_kpc3: float, v_kms: float, t_gyr: float) -> float:
        rho_cgs = rho_msun_per_kpc3 * PC.Msun_to_g / (PC.kpc_to_cm**3)
        sigma_cgs = self.sigma_over_m(v_kms)
        v_cgs = v_kms * PC.km_to_cm
        t_s = t_gyr * PC.Gyr_to_s
        return rho_cgs * sigma_cgs * v_cgs * t_s
    
    def thermalization_radius(self, rho_s: float, r_s: float, 
                              Mhalo: float, t_age: float) -> float:
        def condition(r):
            if r <= 0:
                return -1e10
            x = r / r_s
            rho = rho_s / (x * (1 + x)**2 + 1e-10)
            M_enc = 4 * np.pi * rho_s * r_s**3 * (np.log(1 + x) - x/(1+x))
            v = np.sqrt(PC.G * M_enc / (r + 1e-10))
            v = np.clip(v, 10, 500)
            tau = self.optical_depth(rho, v, t_age)
            return tau - 1.0
        
        try:
            r_min, r_max = 0.01 * r_s, 5.0 * r_s
            f_min, f_max = condition(r_min), condition(r_max)
            if f_min * f_max > 0:
                return 0.5 * r_s if f_min < 0 else 2.0 * r_s
            return np.clip(brentq(condition, r_min, r_max), 0.05, 20.0)
        except:
            return 0.5 * r_s
    
    def core_density(self, rho_s: float, r_s: float, r_1: float, Mhalo: float) -> float:
        x_1 = r_1 / r_s
        M_nfw_r1 = 4 * np.pi * rho_s * r_s**3 * (np.log(1 + x_1) - x_1 / (1 + x_1))
        rho_0 = M_nfw_r1 / (4/3 * np.pi * r_1**3) * 1.5
        return min(rho_0, 10 * rho_s)
    
    def density_profile(self, r: np.ndarray, rho_s: float, r_s: float,
                        Mhalo: float, Mstar: float = 0) -> Tuple[np.ndarray, float]:
        t_age = ScalingRelations.halo_age(Mhalo)
        r_1 = self.thermalization_radius(rho_s, r_s, Mhalo, t_age)
        
        if Mstar > 0:
            f_bar = np.clip(Mstar / Mhalo / 0.03, 0.3, 3.0)
            r_1 *= f_bar**(-0.3)
        
        r_core = r_1
        rho_0 = self.core_density(rho_s, r_s, r_1, Mhalo)
        
        rho_isothermal = rho_0 / (1 + (r / r_core)**2)
        x = r / r_s
        rho_nfw = rho_s / (x * (1 + x)**2 + 1e-10)
        
        r_match = 2.0 * r_core
        transition = 0.5 * (1 + np.tanh((r - r_match) / (0.5 * r_core)))
        rho = (1 - transition) * rho_isothermal + transition * rho_nfw
        
        return rho, r_core
    
    def rotation_curve(self, r: np.ndarray, rho_s: float, r_s: float,
                       Mhalo: float, Mstar: float = 0) -> Tuple[np.ndarray, float]:
        r_fine = np.logspace(np.log10(max(r.min(), 0.01)), 
                             np.log10(r.max() * 1.5), 500)
        rho, r_core = self.density_profile(r_fine, rho_s, r_s, Mhalo, Mstar)
        integrand = 4 * np.pi * r_fine**2 * rho
        M_enc = cumulative_trapezoid(integrand, r_fine, initial=0)
        if len(M_enc) > 1:
            M_enc[0] = M_enc[1] * (r_fine[0] / r_fine[1])**3
        V_fine = np.sqrt(PC.G * np.maximum(M_enc, 0) / np.maximum(r_fine, 0.01))
        V_dm = np.interp(r, r_fine, V_fine)
        return V_dm, r_core


def disk_velocity_freeman(r, M_disk, R_disk):
    y = np.clip(r / (2.0 * R_disk), 1e-10, 100)
    Sigma_0 = M_disk / (2.0 * np.pi * R_disk**2)
    bessel_term = i0(y) * k0(y) - i1(y) * k1(y)
    V2 = 4.0 * np.pi * PC.G * Sigma_0 * R_disk * y**2 * bessel_term
    return np.sqrt(np.maximum(V2, 0))


# =============================================================================
# GÉNÉRATION MOCK
# =============================================================================
def generate_mock_galaxy(name, Vflat_target, sidm_model, Y_disk_true=0.5, 
                         Y_bulge_true=0.7, n_points=25, error_frac=0.08, seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    Mhalo = ScalingRelations.Mhalo_from_Vflat(Vflat_target)
    rho_s_true = ScalingRelations.rho_s(Mhalo)
    r_s_true = ScalingRelations.r_s(Mhalo)
    
    f_bar = np.clip(0.03 * (Mhalo / 1e12)**0.2, 0.001, 0.05)
    Mstar = Mhalo * f_bar
    L_disk = Mstar / Y_disk_true * 0.9
    L_bulge = Mstar / Y_bulge_true * 0.1
    
    R_disk = np.clip(0.015 * ScalingRelations.r_vir(Mhalo), 0.5, 15.0)
    R_bulge = R_disk * 0.2
    
    f_gas = np.clip(0.3 * (Mhalo / 1e11)**(-0.3), 0.05, 0.8)
    M_gas = Mstar * f_gas
    R_gas = R_disk * 1.5
    
    r_max = max(min(8 * R_disk, 5 * r_s_true), 5.0)
    r = np.linspace(0.3, r_max, n_points)
    
    # Vdisk et Vbul pour M/L = 1 (comme SPARC)
    Vdisk_ml1 = disk_velocity_freeman(r, L_disk, R_disk)
    Vbul_ml1 = disk_velocity_freeman(r, L_bulge, R_bulge)
    Vgas = disk_velocity_freeman(r, M_gas, R_gas)
    
    # Vraie courbe avec les vrais M/L
    Vbar = np.sqrt(Vgas**2 + Y_disk_true * Vdisk_ml1**2 + Y_bulge_true * Vbul_ml1**2)
    V_dm, r_core = sidm_model.rotation_curve(r, rho_s_true, r_s_true, Mhalo, Mstar)
    V_true = np.sqrt(Vbar**2 + V_dm**2)
    
    # Erreur minimale de 2 km/s comme dans SPARC
    V_err = np.maximum(V_true * error_frac, 2.0)
    V_obs = np.maximum(V_true + np.random.normal(0, V_err), 5.0)
    
    galaxy = GalaxyData(
        name=name, radius=r, Vobs=V_obs, errV=V_err,
        Vgas=Vgas, Vdisk=Vdisk_ml1, Vbul=Vbul_ml1,
        luminosity=L_disk + L_bulge,
        Y_disk_true=Y_disk_true, Y_bulge_true=Y_bulge_true
    )
    
    true_params = {
        'log_rho_s': np.log10(rho_s_true), 'log_r_s': np.log10(r_s_true),
        'Y_disk': Y_disk_true, 'Y_bulge': Y_bulge_true, 'Mhalo': Mhalo,
        'r_core': r_core, 'V_true': V_true, 'V_dm': V_dm
    }
    
    return galaxy, true_params


def create_mock_sample(sigma_0_true, v_0_true, n_galaxies=80):
    """Créer un échantillon stratifié comme dans SPARC"""
    print(f"\n{'='*60}")
    print(f"GÉNÉRATION DE {n_galaxies} GALAXIES MOCK")
    print(f"σ₀ = {sigma_0_true:.2f} cm²/g, v₀ = {v_0_true:.1f} km/s")
    print(f"{'='*60}")
    
    sidm = SIDMModel(sigma_0_true, v_0_true)
    
    # Distribution de Vflat similaire à SPARC (35-383 km/s)
    np.random.seed(42)
    Vflat_targets = np.concatenate([
        np.linspace(35, 60, 20),    # Naines
        np.linspace(65, 120, 25),   # LSB
        np.linspace(125, 200, 20),  # Spirales
        np.linspace(210, 350, 15),  # Massives
    ])[:n_galaxies]
    
    # Varier les M/L comme dans la réalité
    Y_disk_values = np.random.uniform(0.3, 0.8, n_galaxies)
    Y_bulge_values = np.random.uniform(0.5, 1.0, n_galaxies)
    
    galaxies, true_params_all = {}, {}
    
    for i, (Vflat, Y_d, Y_b) in enumerate(zip(Vflat_targets, Y_disk_values, Y_bulge_values)):
        name = f"Mock_{i+1:03d}_V{Vflat:.0f}"
        galaxy, true_params = generate_mock_galaxy(
            name, Vflat, sidm,
            Y_disk_true=Y_d,
            Y_bulge_true=Y_b,
            n_points=np.random.randint(15, 50),
            error_frac=np.random.uniform(0.05, 0.12),
            seed=42+i
        )
        galaxies[name] = galaxy
        true_params_all[name] = true_params
    
    Vmax_values = [g.Vmax for g in galaxies.values()]
    print(f"  Vmax: {min(Vmax_values):.0f} - {max(Vmax_values):.0f} km/s")
    print(f"  Galaxies < v₀: {sum(1 for v in Vmax_values if v < v_0_true)}")
    print(f"  Galaxies > v₀: {sum(1 for v in Vmax_values if v > v_0_true)}")
    
    return galaxies, true_params_all


# =============================================================================
# FIT AVEC M/L LIBRES (comme dans le code SPARC)
# =============================================================================
def chi2_single_galaxy_free_ML(params, galaxy_dict, sigma_0, v_0):
    """
    CORRECTION MAJEURE: Les M/L sont maintenant des paramètres libres,
    exactement comme dans le code SPARC principal.
    
    params = [log_rho_s, log_r_s, Y_disk, Y_bulge]
    """
    log_rho_s, log_r_s, Y_disk, Y_bulge = params
    
    # Bornes (comme dans GalaxyFitter.get_bounds())
    if not (4.0 < log_rho_s < 10.0):
        return 1e10
    if not (-1.5 < log_r_s < 2.5):
        return 1e10
    if not (0.1 <= Y_disk <= 1.2):
        return 1e10
    if not (0.3 <= Y_bulge <= 1.5):
        return 1e10
    
    rho_s = 10**log_rho_s
    r_s = 10**log_r_s
    
    r = galaxy_dict['radius']
    Vobs = galaxy_dict['Vobs']
    errV = galaxy_dict['errV']
    Vgas = galaxy_dict['Vgas']
    Vdisk = galaxy_dict['Vdisk']
    Vbul = galaxy_dict['Vbul']
    Vflat = galaxy_dict['Vflat']
    
    Mhalo = ScalingRelations.Mhalo_from_Vflat(Vflat)
    Mstar = Y_disk * Vdisk[-1]**2 * r[-1] / PC.G * 1.5 if Vdisk[-1] > 0 else 0
    
    # Composante baryonique avec les M/L fittés
    V_bar = np.sqrt(Vgas**2 + Y_disk * Vdisk**2 + Y_bulge * Vbul**2)
    
    sidm = SIDMModel(sigma_0, v_0)
    try:
        V_dm, _ = sidm.rotation_curve(r, rho_s, r_s, Mhalo, Mstar)
    except:
        return 1e10
    
    V_model = np.sqrt(V_bar**2 + V_dm**2)
    chi2 = np.sum(((Vobs - V_model) / errV)**2)
    
    return chi2 if np.isfinite(chi2) else 1e10


def fit_single_galaxy_worker(args):
    """Worker pour le fit d'une galaxie avec M/L libres"""
    galaxy_dict, sigma_0, v_0 = args
    
    Vflat = galaxy_dict['Vflat']
    Mhalo = ScalingRelations.Mhalo_from_Vflat(Vflat)
    rho_s_est = ScalingRelations.rho_s(Mhalo)
    r_s_est = ScalingRelations.r_s(Mhalo)
    
    # 4 paramètres libres comme dans SPARC
    bounds = [
        (np.log10(rho_s_est) - 2, np.log10(rho_s_est) + 2),
        (np.log10(r_s_est) - 1.5, np.log10(r_s_est) + 1.5),
        (0.1, 1.2),   # Y_disk
        (0.3, 1.5),   # Y_bulge
    ]
    
    try:
        result = differential_evolution(
            lambda p: chi2_single_galaxy_free_ML(p, galaxy_dict, sigma_0, v_0),
            bounds,
            seed=42,
            maxiter=500,
            tol=1e-5,
            polish=True,
            workers=1
        )
        n_params = 4  # log_rho_s, log_r_s, Y_disk, Y_bulge
        n_dof = galaxy_dict['n_points'] - n_params
        return galaxy_dict['name'], result.x, result.fun, n_dof, result.success
    except Exception as e:
        return galaxy_dict['name'], None, 1e10, 0, False


def galaxy_to_dict(galaxy):
    return {
        'name': galaxy.name,
        'radius': galaxy.radius,
        'Vobs': galaxy.Vobs,
        'errV': galaxy.errV,
        'Vgas': galaxy.Vgas,
        'Vdisk': galaxy.Vdisk,
        'Vbul': galaxy.Vbul,
        'Vflat': galaxy.Vflat,
        'n_points': galaxy.n_points,
        'Y_disk_true': galaxy.Y_disk_true,
        'Y_bulge_true': galaxy.Y_bulge_true,
    }


def global_chi2_parallel(log_params, galaxy_dicts, n_workers=None):
    """
    Fonction objectif globale: χ² global / dof global
    Exactement comme dans GlobalSIDMOptimizer.objective()
    """
    sigma_0 = 10**log_params[0]
    v_0 = 10**log_params[1]
    
    if n_workers is None:
        n_workers = N_WORKERS
    
    args_list = [(gd, sigma_0, v_0) for gd in galaxy_dicts]
    
    chi2_total = 0.0
    dof_total = 0
    n_success = 0
    
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        results = list(executor.map(fit_single_galaxy_worker, args_list))
    
    for name, params, chi2, n_dof, success in results:
        if success and chi2 < 1e9:
            chi2_total += chi2
            dof_total += n_dof
            n_success += 1
    
    # Comme dans le code SPARC: soustraire les 2 paramètres globaux
    n_global_params = 2
    dof_effective = dof_total - n_global_params
    
    if dof_effective <= 0 or n_success < len(galaxy_dicts) * 0.3:
        return 1e6
    
    chi2_red_global = chi2_total / dof_effective
    
    return chi2_red_global


# =============================================================================
# OPTIMISATION GLOBALE
# =============================================================================
def fit_global_parameters(galaxies, verbose=True):
    """
    Optimisation globale de σ₀ et v₀
    Réplique exacte de GlobalSIDMOptimizer
    """
    if verbose:
        print(f"\n{'='*60}")
        print("OPTIMISATION GLOBALE PARALLÉLISÉE")
        print(f"N_WORKERS = {N_WORKERS}")
        print(f"{'='*60}")
    
    galaxy_dicts = [galaxy_to_dict(g) for g in galaxies.values()]
    
    # Bornes comme dans le code principal
    bounds = [(0.5, 2.5), (1.2, 2.5)]  # log10(σ₀), log10(v₀)
    
    best_result = None
    best_chi2 = 1e10
    
    for iteration in range(3):
        if verbose:
            print(f"\n[Iteration {iteration+1}/3]")
        
        result = differential_evolution(
            lambda p: global_chi2_parallel(p, galaxy_dicts),
            bounds,
            strategy='best1bin',
            popsize=15,
            maxiter=50,
            workers=1,  # Parallelisation au niveau des galaxies
            seed=42 + iteration,
            tol=0.02,
            disp=False,
            polish=True
        )
        
        if result.fun < best_chi2:
            best_chi2 = result.fun
            best_result = result
            if verbose:
                sigma_0 = 10**result.x[0]
                v_0 = 10**result.x[1]
                print(f"  Nouveau min: σ₀={sigma_0:.2f}, v₀={v_0:.1f}, χ²/dof={best_chi2:.4f}")
    
    sigma_0 = 10**best_result.x[0]
    v_0 = 10**best_result.x[1]
    
    if verbose:
        print(f"\n  Résultat final: σ₀={sigma_0:.2f} cm²/g, v₀={v_0:.1f} km/s")
        print(f"  χ²/dof = {best_chi2:.4f}")
    
    return sigma_0, v_0, best_chi2


# =============================================================================
# SCAN DU PAYSAGE χ²
# =============================================================================
def scan_single_point(args):
    sigma_0, v_0, galaxy_dicts = args
    chi2 = global_chi2_parallel([np.log10(sigma_0), np.log10(v_0)], galaxy_dicts, n_workers=1)
    return sigma_0, v_0, chi2


def scan_chi2_landscape(galaxies, sigma_range, v0_range, n_workers=None):
    """Scan 2D du paysage χ²"""
    print(f"\n{'='*60}")
    print("SCAN DU PAYSAGE χ² 2D")
    print(f"{'='*60}")
    
    if n_workers is None:
        n_workers = N_WORKERS
    
    galaxy_dicts = [galaxy_to_dict(g) for g in galaxies.values()]
    points = [(s, v, galaxy_dicts) for s in sigma_range for v in v0_range]
    
    print(f"  {len(points)} points à évaluer sur {n_workers} workers...")
    t0 = time.time()
    
    chi2_map = np.zeros((len(sigma_range), len(v0_range)))
    
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(scan_single_point, p): p for p in points}
        
        done = 0
        for future in as_completed(futures):
            sigma_0, v_0, chi2 = future.result()
            i = np.where(sigma_range == sigma_0)[0][0]
            j = np.where(v0_range == v_0)[0][0]
            chi2_map[i, j] = chi2
            
            done += 1
            if done % 20 == 0:
                elapsed = time.time() - t0
                eta = elapsed / done * (len(points) - done)
                print(f"  {done}/{len(points)} done, ETA: {eta/60:.1f} min")
    
    print(f"  Terminé en {(time.time()-t0)/60:.1f} min")
    
    return chi2_map


# =============================================================================
# VISUALISATION
# =============================================================================
def plot_comprehensive_results(galaxies, true_params_all, 
                                sigma_0_fit, v_0_fit, chi2_fit,
                                sigma_0_true, v_0_true,
                                chi2_map=None, sigma_range=None, v0_range=None,
                                filename="mock_test_results.png"):
    """Figure complète de diagnostic"""
    
    fig = plt.figure(figsize=(20, 16))
    
    # ==========================================================================
    # Row 1: Courbes de rotation représentatives
    # ==========================================================================
    sidm_fit = SIDMModel(sigma_0_fit, v_0_fit)
    
    names = list(galaxies.keys())
    Vmax_sorted = sorted(names, key=lambda n: galaxies[n].Vmax)
    indices = [0, len(Vmax_sorted)//4, len(Vmax_sorted)//2, 3*len(Vmax_sorted)//4, -1]
    selected = [Vmax_sorted[i] for i in indices][:5]
    
    for i, name in enumerate(selected):
        ax = fig.add_subplot(4, 5, i + 1)
        galaxy = galaxies[name]
        true_p = true_params_all[name]
        r = galaxy.radius
        
        ax.errorbar(r, galaxy.Vobs, yerr=galaxy.errV, fmt='ko', ms=3, capsize=1, label='Data')
        ax.plot(r, true_p['V_true'], 'b-', lw=2, alpha=0.7, label='True')
        
        ax.set_title(f"Vmax={galaxy.Vmax:.0f} km/s", fontsize=9)
        ax.set_xlabel('R (kpc)', fontsize=8)
        ax.set_ylabel('V (km/s)', fontsize=8)
        if i == 0:
            ax.legend(fontsize=6, loc='lower right')
    
    # ==========================================================================
    # Row 2: Paysage χ² et diagnostics
    # ==========================================================================
    
    # Paysage χ² 2D
    ax = fig.add_subplot(4, 5, 6)
    if chi2_map is not None:
        chi2_min = chi2_map.min()
        delta_chi2 = chi2_map - chi2_min
        
        im = ax.contourf(v0_range, sigma_range, delta_chi2,
                        levels=[0, 1, 2.3, 4, 6.17, 9, 15, 25], cmap='RdYlBu_r')
        ax.contour(v0_range, sigma_range, delta_chi2,
                  levels=[2.3, 6.17], colors='black', linewidths=[1, 1.5])
        ax.plot(v_0_true, sigma_0_true, 'g*', ms=15, label='True', zorder=10)
        ax.plot(v_0_fit, sigma_0_fit, 'rx', ms=12, mew=3, label='Fit', zorder=10)
        ax.set_xlabel('v₀ (km/s)', fontsize=10)
        ax.set_ylabel('σ₀ (cm²/g)', fontsize=10)
        ax.set_title('Paysage χ² (Δχ²)', fontsize=10)
        ax.legend(fontsize=8)
        plt.colorbar(im, ax=ax, label='Δχ²')
    
    # Section efficace
    ax = fig.add_subplot(4, 5, 7)
    v = np.logspace(1, 2.7, 100)
    sidm_true = SIDMModel(sigma_0_true, v_0_true)
    ax.loglog(v, sidm_true.sigma_over_m(v), 'b-', lw=3, label=f'True (σ₀={sigma_0_true:.1f}, v₀={v_0_true:.0f})')
    ax.loglog(v, sidm_fit.sigma_over_m(v), 'r--', lw=3, label=f'Fit (σ₀={sigma_0_fit:.1f}, v₀={v_0_fit:.0f})')
    
    Vmax_all = [g.Vmax for g in galaxies.values()]
    ax.axvspan(min(Vmax_all), max(Vmax_all), alpha=0.15, color='orange', label='Sample Vmax')
    ax.set_xlabel('v (km/s)', fontsize=10)
    ax.set_ylabel('σ/m (cm²/g)', fontsize=10)
    ax.set_title('Section Efficace', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    
    # Distribution Vmax
    ax = fig.add_subplot(4, 5, 8)
    ax.hist(Vmax_all, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(v_0_true, color='blue', ls='--', lw=2, label=f'v₀ true = {v_0_true:.0f}')
    ax.axvline(v_0_fit, color='red', ls=':', lw=2, label=f'v₀ fit = {v_0_fit:.0f}')
    ax.set_xlabel('Vmax (km/s)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title('Distribution Vmax', fontsize=10)
    ax.legend(fontsize=8)
    
    # Comparaison σ/m
    ax = fig.add_subplot(4, 5, 9)
    v_test = np.array([30, 50, 80, 120, 200])
    sigma_true_vals = sidm_true.sigma_over_m(v_test)
    sigma_fit_vals = sidm_fit.sigma_over_m(v_test)
    ratio = sigma_fit_vals / sigma_true_vals
    
    ax.bar(range(len(v_test)), ratio, color='coral', edgecolor='black')
    ax.axhline(1, color='green', ls='--', lw=2)
    ax.set_xticks(range(len(v_test)))
    ax.set_xticklabels([f'{v:.0f}' for v in v_test])
    ax.set_xlabel('v (km/s)', fontsize=10)
    ax.set_ylabel('σ_fit / σ_true', fontsize=10)
    ax.set_title('Ratio σ/m (fit/true)', fontsize=10)
    ax.set_ylim(0, 2)
    
    # Coupe le long de la vallée
    ax = fig.add_subplot(4, 5, 10)
    if chi2_map is not None:
        min_chi2_per_sigma = chi2_map.min(axis=1)
        ax.plot(sigma_range, min_chi2_per_sigma, 'b-', lw=2)
        ax.axvline(sigma_0_true, color='green', ls='--', lw=2, label='σ₀ true')
        ax.axvline(sigma_0_fit, color='red', ls=':', lw=2, label='σ₀ fit')
        ax.set_xlabel('σ₀ (cm²/g)', fontsize=10)
        ax.set_ylabel('min χ²/dof', fontsize=10)
        ax.set_title('Coupe χ² (marginalisant v₀)', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # ==========================================================================
    # Row 3: Plus de diagnostics
    # ==========================================================================
    
    # v₀ optimal pour chaque σ₀
    ax = fig.add_subplot(4, 5, 11)
    if chi2_map is not None:
        best_v0_per_sigma = v0_range[chi2_map.argmin(axis=1)]
        ax.plot(sigma_range, best_v0_per_sigma, 'purple', lw=2, marker='o')
        ax.axhline(v_0_true, color='green', ls='--', lw=2, label='v₀ true')
        ax.axvline(sigma_0_true, color='blue', ls=':', lw=2, label='σ₀ true')
        ax.set_xlabel('σ₀ (cm²/g)', fontsize=10)
        ax.set_ylabel('v₀ optimal (km/s)', fontsize=10)
        ax.set_title('Dégénérescence σ₀-v₀', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Erreur relative sur σ/m vs v
    ax = fig.add_subplot(4, 5, 12)
    v_range = np.linspace(20, 300, 50)
    sigma_true = sidm_true.sigma_over_m(v_range)
    sigma_fit = sidm_fit.sigma_over_m(v_range)
    error_pct = 100 * (sigma_fit - sigma_true) / sigma_true
    
    ax.plot(v_range, error_pct, 'crimson', lw=2)
    ax.axhline(0, color='black', ls='-', lw=1)
    ax.axhline(20, color='gray', ls='--', alpha=0.5)
    ax.axhline(-20, color='gray', ls='--', alpha=0.5)
    ax.fill_between([min(Vmax_all), max(Vmax_all)], -100, 100, alpha=0.1, color='orange')
    ax.set_xlabel('v (km/s)', fontsize=10)
    ax.set_ylabel('Erreur σ/m (%)', fontsize=10)
    ax.set_title('Erreur relative sur σ/m', fontsize=10)
    ax.set_ylim(-50, 50)
    ax.grid(True, alpha=0.3)
    
    # ==========================================================================
    # Row 4: Résumé et conclusions
    # ==========================================================================
    
    ax = fig.add_subplot(4, 5, 16)
    ax.axis('off')
    
    delta_sigma = 100 * (sigma_0_fit - sigma_0_true) / sigma_0_true
    delta_v = 100 * (v_0_fit - v_0_true) / v_0_true
    
    # Erreur sur σ/m à différentes vitesses
    err_30 = 100 * (sidm_fit.sigma_over_m(30) - sidm_true.sigma_over_m(30)) / sidm_true.sigma_over_m(30)
    err_100 = 100 * (sidm_fit.sigma_over_m(100) - sidm_true.sigma_over_m(100)) / sidm_true.sigma_over_m(100)
    err_200 = 100 * (sidm_fit.sigma_over_m(200) - sidm_true.sigma_over_m(200)) / sidm_true.sigma_over_m(200)
    
    success = abs(delta_sigma) < 30 and abs(delta_v) < 30
    success_physics = abs(err_30) < 30 and abs(err_100) < 50
    
    summary = f"""
RÉSUMÉ MOCK TEST (M/L LIBRES)
=============================

VALEURS VRAIES:
  σ₀ = {sigma_0_true:.2f} cm²/g
  v₀ = {v_0_true:.1f} km/s

VALEURS FITTÉES:
  σ₀ = {sigma_0_fit:.2f} cm²/g [{delta_sigma:+.1f}%]
  v₀ = {v_0_fit:.1f} km/s [{delta_v:+.1f}%]
  χ²/dof = {chi2_fit:.3f}

ERREUR SUR σ/m(v):
  @ 30 km/s:  {err_30:+.1f}%
  @ 100 km/s: {err_100:+.1f}%
  @ 200 km/s: {err_200:+.1f}%

N galaxies = {len(galaxies)}
    """
    
    color = 'lightgreen' if success_physics else 'lightsalmon'
    ax.text(0.05, 0.95, summary, transform=ax.transAxes, fontsize=10,
           fontfamily='monospace', verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))
    
    # Conclusions
    ax = fig.add_subplot(4, 5, 17)
    ax.axis('off')
    
    conclusion = f"""
CONCLUSION
==========

"""
    if success_physics:
        conclusion += """
✓ Le modèle récupère correctement
  la physique (σ/m vs v)

Les paramètres (σ₀, v₀) peuvent varier
mais la courbe σ/m(v) reste proche
dans la plage de vitesses sondée.
"""
    else:
        conclusion += """
✗ Divergence significative

Possible dégénérescence σ₀-v₀
ou problème dans le modèle SIDM.
"""
    
    ax.text(0.05, 0.95, conclusion, transform=ax.transAxes, fontsize=10,
           fontfamily='monospace', verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n✓ Figure sauvegardée: {os.path.join(OUTPUT_DIR, filename)}")


# =============================================================================
# MAIN
# =============================================================================
def main():
    print("="*70)
    print("SIDM MOCK TEST - VERSION CORRIGÉE (M/L LIBRES)")
    print(f"N_WORKERS = {N_WORKERS}")
    print("="*70)
    
    # Paramètres vrais = ceux trouvés sur SPARC
    SIGMA_0_TRUE = 3.27
    V_0_TRUE = 34.9
    N_GALAXIES = 80  # Comme dans SPARC
    
    t_total = time.time()
    
    # 1. Génération des données mock
    galaxies, true_params = create_mock_sample(SIGMA_0_TRUE, V_0_TRUE, N_GALAXIES)
    
    # 2. Scan 2D
    print("\n[1/3] Scan 2D du paysage χ²...")
    sigma_range = np.array([1, 1.5, 2, 2.5, 3, 3.5, 4, 5, 6, 8, 10])
    v0_range = np.array([20, 25, 30, 35, 40, 45, 50, 60, 70, 80])
    chi2_map = scan_chi2_landscape(galaxies, sigma_range, v0_range)
    
    # 3. Optimisation globale
    print("\n[2/3] Optimisation globale...")
    sigma_0_fit, v_0_fit, chi2_fit = fit_global_parameters(galaxies)
    
    # 4. Résultats
    delta_s = 100 * (sigma_0_fit - SIGMA_0_TRUE) / SIGMA_0_TRUE
    delta_v = 100 * (v_0_fit - V_0_TRUE) / V_0_TRUE
    
    print(f"\n{'='*70}")
    print("RÉSUMÉ FINAL")
    print(f"{'='*70}")
    print(f"\nValeurs vraies:")
    print(f"  σ₀ = {SIGMA_0_TRUE:.2f} cm²/g")
    print(f"  v₀ = {V_0_TRUE:.1f} km/s")
    print(f"\nValeurs fittées:")
    print(f"  σ₀ = {sigma_0_fit:.2f} cm²/g [{delta_s:+.1f}%]")
    print(f"  v₀ = {v_0_fit:.1f} km/s [{delta_v:+.1f}%]")
    print(f"  χ²/dof = {chi2_fit:.4f}")
    print(f"\nTemps total: {(time.time()-t_total)/60:.1f} min")
    
    # 5. Visualisation
    print("\n[3/3] Génération de la figure...")
    plot_comprehensive_results(
        galaxies, true_params,
        sigma_0_fit, v_0_fit, chi2_fit,
        SIGMA_0_TRUE, V_0_TRUE,
        chi2_map, sigma_range, v0_range
    )
    
    # 6. Sauvegarder les résultats
    results = {
        'sigma_0_true': SIGMA_0_TRUE,
        'v_0_true': V_0_TRUE,
        'sigma_0_fit': sigma_0_fit,
        'v_0_fit': v_0_fit,
        'chi2_fit': chi2_fit,
        'bias_sigma_pct': delta_s,
        'bias_v_pct': delta_v,
        'n_galaxies': N_GALAXIES,
        'success': abs(delta_s) < 30 and abs(delta_v) < 30
    }
    
    with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n{'='*70}")
    if results['success']:
        print("✓ MOCK TEST RÉUSSI - Les paramètres sont bien récupérés")
    else:
        print("⚠ ATTENTION - Déviation significative détectée")
    print(f"{'='*70}")
    
    return results


if __name__ == "__main__":
    results = main()